In [18]:
import numpy as np
import pandas as pd
from scipy import stats

In [19]:
validator_exits_size = pd.read_csv('../int/validator_exits_size.csv')
validator_exits_category = pd.read_csv('../int/validator_exits_category.csv')
validator_exits_pool = pd.read_csv('../int/validator_exits_pool.csv')
active_validators_size = pd.read_csv('../int/active_validators_size.csv')
active_validators_category = pd.read_csv('../int/active_validators_category.csv')
active_validators_pool = pd.read_csv('../int/active_validators_pool.csv')

In [20]:
validator_exits_size

,slot,1,100+,2-5,20-99,6-19,total
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2.0,0.0,0.0,0.0,0.0,0.0,0.0
3,3.0,0.0,0.0,0.0,0.0,0.0,0.0
4,4.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...
8986171,8986171.0,0.0,0.0,0.0,0.0,0.0,0.0
8986172,8986172.0,0.0,0.0,0.0,0.0,0.0,0.0
8986173,8986173.0,0.0,0.0,0.0,0.0,0.0,0.0
8986174,8986174.0,0.0,0.0,0.0,0.0,0.0,0.0


In [21]:
aave = pd.read_csv('../int/aave_grouped.csv', usecols=('slot', 'liquidity_apr'))
aave['price_pct_change'] = aave['liquidity_apr'].pct_change()
aave = aave.dropna()
aave = aave[aave['slot'].isin(validator_exits_size['slot'])]
aave = aave[aave['slot'] >= 6206400]
aave

/var/folders/mb/5hm6pgrs3zj_1m_kgvpt40jw0000gn/T/ipykernel_24043/3588217129.py:2: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  aave['price_pct_change'] = aave['liquidity_apr'].pct_change()


,slot,liquidity_apr,price_pct_change
75,6206400.0,2.301241,0.010450
76,6213600.0,2.441569,0.060979
77,6220800.0,2.242568,-0.081506
78,6228000.0,2.073961,-0.075185
79,6235200.0,1.928353,-0.070208
...,...,...,...
457,8956800.0,1.371801,-0.037020
458,8964000.0,1.377068,0.003839
459,8971200.0,1.370012,-0.005124
460,8978400.0,1.308489,-0.044907


In [22]:
validator_exits_size_set = validator_exits_size
validator_exits_size_set['slot'] = validator_exits_size_set['slot'] // 300 * 300

# Group by the day and get the last slot of each day and sum values for all columns
validator_exits_size_set = validator_exits_size_set.groupby('slot').agg(
    {
        'slot': 'last',
        '1': 'sum',
        '2-5': 'sum',
        '6-19': 'sum',
        '20-99': 'sum',
        '100+': 'sum',
        'total': 'sum'
    }
).reset_index(drop=True)

validator_exits_size_set


,slot,1,2-5,6-19,20-99,100+,total
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,300.0,0.0,0.0,0.0,0.0,0.0,0.0
2,600.0,0.0,0.0,0.0,0.0,0.0,0.0
3,900.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1200.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...
29949,8984700.0,3.0,0.0,0.0,0.0,83.0,86.0
29950,8985000.0,0.0,0.0,0.0,0.0,12.0,12.0
29951,8985300.0,0.0,0.0,0.0,0.0,6.0,6.0
29952,8985600.0,1.0,0.0,0.0,0.0,0.0,1.0


In [23]:
validator_exits_category_set = validator_exits_category
validator_exits_category_set['slot'] = validator_exits_category_set['slot'] // 300 * 300

# Create a dictionary for aggregation
agg_dict = {col: 'sum' for col in validator_exits_category_set.columns if col != 'slot'}
agg_dict['slot'] = 'last'

# Group by 'slot' and apply the aggregation
validator_exits_category_set = validator_exits_category_set.groupby('slot').agg(agg_dict).reset_index(drop=True)

validator_exits_category_set

,CEX,Liquid Restaking,Liquid Staking,Solo Stakers,Staking Pools,Unidentified,total,slot
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,300.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,600.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,900.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1200.0
...,...,...,...,...,...,...,...,...
29949,12.0,0.0,4.0,0.0,2.0,68.0,86.0,8984700.0
29950,0.0,0.0,3.0,0.0,9.0,0.0,12.0,8985000.0
29951,0.0,0.0,4.0,0.0,0.0,2.0,6.0,8985300.0
29952,0.0,0.0,0.0,0.0,0.0,1.0,1.0,8985600.0


In [24]:
validator_exits_pool_set = validator_exits_pool
validator_exits_pool_set['slot'] = validator_exits_pool_set['slot'] // 300 * 300

# Create a dictionary for aggregation
agg_dict = {col: 'sum' for col in validator_exits_pool_set.columns if col != 'slot'}
agg_dict['slot'] = 'last'

# Group by 'slot' and apply the aggregation
validator_exits_pool_set = validator_exits_pool_set.groupby('slot').agg(agg_dict).reset_index(drop=True)

validator_exits_pool_set

,Binance,Bitcoin Suisse,Coinbase,Ether.Fi,Kraken,Ledger Live,Lido,Mantle,OKX,Other Stakers,Rocketpool,total,slot
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,300.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,600.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,900.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1200.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
29949,0.0,0.0,9.0,0.0,0.0,0.0,0.0,2.0,0.0,73.0,2.0,86.0,8984700.0
29950,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.0,3.0,12.0,8985000.0
29951,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,4.0,6.0,8985300.0
29952,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,8985600.0


In [25]:
active_validators_size_set = active_validators_size[active_validators_size['slot'].isin(validator_exits_size_set['slot'])]
active_validators_size_set.reset_index(inplace=True)
active_validators_size_set = active_validators_size_set.drop(columns=['index'])
active_validators_size_set

,slot,1,100+,2-5,20-99,6-19,total
0,0.0,1020.0,13599.0,1049.0,3568.0,1827.0,21063.0
1,300.0,1020.0,13599.0,1049.0,3568.0,1827.0,21063.0
2,600.0,1020.0,13599.0,1049.0,3568.0,1827.0,21063.0
3,900.0,1020.0,13599.0,1049.0,3568.0,1827.0,21063.0
4,1200.0,1020.0,13599.0,1049.0,3568.0,1827.0,21063.0
...,...,...,...,...,...,...,...
29949,8984700.0,9846.0,919817.0,9925.0,44358.0,18082.0,1002028.0
29950,8985000.0,9843.0,919814.0,9925.0,44358.0,18082.0,1002022.0
29951,8985300.0,9845.0,919872.0,9925.0,44358.0,18082.0,1002082.0
29952,8985600.0,9845.0,919946.0,9925.0,44358.0,18082.0,1002156.0


In [26]:
active_validators_category_set = active_validators_category[active_validators_category['slot'].isin(validator_exits_category_set['slot'])]
active_validators_category_set.reset_index(inplace=True)
active_validators_category_set = active_validators_category_set.drop(columns=['index'])
active_validators_category_set

,slot,CEX,Liquid Restaking,Liquid Staking,Solo Stakers,Staking Pools,Unidentified,total
0,0.0,3468.0,0.0,559.0,2309.0,622.0,14105.0,21063.0
1,300.0,3468.0,0.0,559.0,2309.0,622.0,14105.0,21063.0
2,600.0,3468.0,0.0,559.0,2309.0,622.0,14105.0,21063.0
3,900.0,3468.0,0.0,559.0,2309.0,622.0,14105.0,21063.0
4,1200.0,3468.0,0.0,559.0,2309.0,622.0,14105.0,21063.0
...,...,...,...,...,...,...,...,...
29949,8984700.0,255920.0,77793.0,331601.0,16837.0,56456.0,263421.0,1002028.0
29950,8985000.0,255908.0,77873.0,331597.0,16837.0,56454.0,263353.0,1002022.0
29951,8985300.0,255908.0,77924.0,331595.0,16837.0,56445.0,263373.0,1002082.0
29952,8985600.0,255908.0,77924.0,331591.0,16837.0,56450.0,263446.0,1002156.0


In [27]:
active_validators_pool_set = active_validators_pool[active_validators_pool['slot'].isin(validator_exits_pool_set['slot'])]
active_validators_pool_set.reset_index(inplace=True)
active_validators_pool_set = active_validators_pool_set.drop(columns=['index'])
active_validators_pool_set

,slot,Binance,Bitcoin Suisse,Coinbase,Ether.Fi,Kraken,Ledger Live,Lido,Mantle,OKX,Other Stakers,Rocketpool,total
0,0.0,1.0,2876.0,0.0,0.0,0.0,13.0,0.0,0.0,0.0,18153.0,20.0,21063.0
1,300.0,1.0,2876.0,0.0,0.0,0.0,13.0,0.0,0.0,0.0,18153.0,20.0,21063.0
2,600.0,1.0,2876.0,0.0,0.0,0.0,13.0,0.0,0.0,0.0,18153.0,20.0,21063.0
3,900.0,1.0,2876.0,0.0,0.0,0.0,13.0,0.0,0.0,0.0,18153.0,20.0,21063.0
4,1200.0,1.0,2876.0,0.0,0.0,0.0,13.0,0.0,0.0,0.0,18153.0,20.0,21063.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
29949,8984700.0,35167.0,19013.0,138349.0,33665.0,23748.0,14723.0,290691.0,14976.0,10877.0,396087.0,24732.0,1002028.0
29950,8985000.0,35167.0,19013.0,138340.0,33745.0,23748.0,14723.0,290691.0,14974.0,10877.0,396014.0,24730.0,1002022.0
29951,8985300.0,35167.0,19013.0,138340.0,33795.0,23748.0,14723.0,290691.0,14974.0,10877.0,396026.0,24728.0,1002082.0
29952,8985600.0,35167.0,19013.0,138340.0,33795.0,23748.0,14728.0,290691.0,14974.0,10877.0,396099.0,24724.0,1002156.0


In [28]:
validator_exits_size_set.set_index('slot', inplace=True)
active_validators_size_set.set_index('slot', inplace=True)
validator_exit_percentage_size = validator_exits_size_set.divide(active_validators_size_set, fill_value=0) * 100
validator_exit_percentage_size.reset_index(inplace=True)
validator_exit_percentage_size

,slot,1,100+,2-5,20-99,6-19,total
0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000000
1,300.0,0.000000,0.000000,0.0,0.0,0.0,0.000000
2,600.0,0.000000,0.000000,0.0,0.0,0.0,0.000000
3,900.0,0.000000,0.000000,0.0,0.0,0.0,0.000000
4,1200.0,0.000000,0.000000,0.0,0.0,0.0,0.000000
...,...,...,...,...,...,...,...
29949,8984700.0,0.030469,0.009024,0.0,0.0,0.0,0.008583
29950,8985000.0,0.000000,0.001305,0.0,0.0,0.0,0.001198
29951,8985300.0,0.000000,0.000652,0.0,0.0,0.0,0.000599
29952,8985600.0,0.010157,0.000000,0.0,0.0,0.0,0.000100


In [29]:
validator_exits_category_set.set_index('slot', inplace=True)
active_validators_category_set.set_index('slot', inplace=True)
validator_exit_percentage_category = validator_exits_category_set.divide(active_validators_category_set, fill_value=0) * 100
validator_exit_percentage_category.reset_index(inplace=True)
validator_exit_percentage_category

,slot,CEX,Liquid Restaking,Liquid Staking,Solo Stakers,Staking Pools,Unidentified,total
0,0.0,0.000000,NaN,0.000000,0.0,0.000000,0.000000,0.000000
1,300.0,0.000000,NaN,0.000000,0.0,0.000000,0.000000,0.000000
2,600.0,0.000000,NaN,0.000000,0.0,0.000000,0.000000,0.000000
3,900.0,0.000000,NaN,0.000000,0.0,0.000000,0.000000,0.000000
4,1200.0,0.000000,NaN,0.000000,0.0,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...
29949,8984700.0,0.004689,0.0,0.001206,0.0,0.003543,0.025814,0.008583
29950,8985000.0,0.000000,0.0,0.000905,0.0,0.015942,0.000000,0.001198
29951,8985300.0,0.000000,0.0,0.001206,0.0,0.000000,0.000759,0.000599
29952,8985600.0,0.000000,0.0,0.000000,0.0,0.000000,0.000380,0.000100


In [30]:
validator_exits_pool_set.set_index('slot', inplace=True)
active_validators_pool_set.set_index('slot', inplace=True)
validator_exit_percentage_pool = validator_exits_pool_set.divide(active_validators_pool_set, fill_value=0) * 100
validator_exit_percentage_pool.reset_index(inplace=True)
validator_exit_percentage_pool

,slot,Binance,Bitcoin Suisse,Coinbase,Ether.Fi,Kraken,Ledger Live,Lido,Mantle,OKX,Other Stakers,Rocketpool,total
0,0.0,0.0,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.000000,0.000000,0.000000
1,300.0,0.0,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.000000,0.000000,0.000000
2,600.0,0.0,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.000000,0.000000,0.000000
3,900.0,0.0,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.000000,0.000000,0.000000
4,1200.0,0.0,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
29949,8984700.0,0.0,0.0,0.006505,0.0,0.0,0.0,0.0,0.013355,0.0,0.018430,0.008087,0.008583
29950,8985000.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.002273,0.012131,0.001198
29951,8985300.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.000505,0.016176,0.000599
29952,8985600.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.000252,0.000000,0.000100


In [31]:
columns = ['1', '100+', '2-5', '20-99', '6-19', 'total']
for col in columns:
    print(validator_exit_percentage_size[col].mean())

0.001797518567377886
0.001577253182078953
0.0013819776968174678
0.0008944387687919705
0.001316424476802735
0.0015420676468236839


In [32]:
# Merge the two DataFrames on the slot column
price_elasticity_size = pd.merge(validator_exit_percentage_size, aave[['slot', 'price_pct_change']], on='slot')

# Drop rows with infinite values
price_elasticity_size.replace([np.inf, -np.inf], np.nan, inplace=True)

# Calculate elasticity for 'total' first
price_elasticity_size['elasticity_total'] = price_elasticity_size['total'] / price_elasticity_size['price_pct_change']
price_elasticity_size['elasticity_total'] = price_elasticity_size['elasticity_total'].replace([np.inf, -np.inf], np.nan)

# Initialize dictionaries to store elasticity values, standard deviations, t-statistics, p-values, and number of valid rows (N)
elasticity = {}
standard_deviations = {}
t_statistics = {}
p_values = {}
valid_counts = {}

# Calculate elasticity for the 'total' column
valid_total_elasticity = price_elasticity_size['elasticity_total'].dropna()
elasticity['total'] = valid_total_elasticity.mean()
standard_deviations['total'] = valid_total_elasticity.std()
t_statistics['total'] = '-'
p_values['total'] = '-'
valid_counts['total'] = len(valid_total_elasticity)

# Calculate elasticity for each column except 'total'
columns = ['1', '2-5', '6-19', '20-99', '100+']
for col in columns:
    price_elasticity_size[f'elasticity_{col}'] = price_elasticity_size[col] / price_elasticity_size['price_pct_change']
    
    # Replace infinite values with NaN
    price_elasticity_size[f'elasticity_{col}'] = price_elasticity_size[f'elasticity_{col}'].replace([np.inf, -np.inf], np.nan)
    
    # Drop NaN values for calculation purposes
    valid_elasticity = price_elasticity_size[f'elasticity_{col}'].dropna()
    
    elasticity[col] = valid_elasticity.mean()
    standard_deviations[col] = valid_elasticity.std()
    
    # Perform a two-sample t-test against the 'total' elasticity
    t_stat, p_value = stats.ttest_ind(valid_elasticity, valid_total_elasticity, equal_var=False)
    t_statistics[col] = t_stat
    p_values[col] = p_value
    
    # Store the number of valid rows
    valid_counts[col] = len(valid_elasticity)

# Print the results in the requested format
print("Elasticity Analysis Results:")
print(f'total: Mean Elasticity = {elasticity["total"]}, t(Mean) = {t_statistics["total"]}, SD = {standard_deviations["total"]}, N = {valid_counts["total"]}, p-value = {p_values["total"]}')
for col in columns:
    print(f'{col}: Mean Elasticity = {elasticity[col]}, t(Mean) = {t_statistics[col]}, SD = {standard_deviations[col]}, N = {valid_counts[col]}, p-value = {p_values[col]}')

# Display the DataFrame with elasticity columns
print(price_elasticity_size)

Elasticity Analysis Results:
total: Mean Elasticity = -0.2077412695628768, t(Mean) = -, SD = 5.70050321762948, N = 387, p-value = -
1: Mean Elasticity = -0.15363229656338795, t(Mean) = 0.16642062237322602, SD = 2.9008288301386513, N = 387, p-value = 0.8678846168744578
2-5: Mean Elasticity = 0.1275422209841679, t(Mean) = 0.8072019497760528, SD = 5.854284948189607, N = 387, p-value = 0.4197989387568859
6-19: Mean Elasticity = 0.3790099896381625, t(Mean) = 1.3410949658855125, SD = 6.448572736122108, N = 387, p-value = 0.1802899789671023
20-99: Mean Elasticity = -0.2984232869813451, t(Mean) = -0.2714245083190339, SD = 3.2713029187451585, N = 387, p-value = 0.7861555835885095
100+: Mean Elasticity = -0.2214139289006326, t(Mean) = -0.0319698246124812, SD = 6.187768739300525, N = 387, p-value = 0.9745044337082773
          slot         1      100+       2-5     20-99      6-19     total  \
0    6206400.0  0.000000  0.015735  0.000000  0.000000  0.000000  0.014214   
1    6213600.0  0.000000  

In [33]:
# Merge the two DataFrames on the slot column
price_elasticity_category = pd.merge(validator_exit_percentage_category, aave[['slot', 'price_pct_change']], on='slot')

# Drop rows with infinite values
price_elasticity_category.replace([np.inf, -np.inf], np.nan, inplace=True)

# Calculate elasticity for 'total' first
price_elasticity_category['elasticity_total'] = price_elasticity_category['total'] / price_elasticity_category['price_pct_change']
price_elasticity_category['elasticity_total'] = price_elasticity_category['elasticity_total'].replace([np.inf, -np.inf], np.nan)

# Initialize dictionaries to store elasticity values, standard deviations, t-statistics, p-values, and number of valid rows (N)
elasticity = {}
standard_deviations = {}
t_statistics = {}
p_values = {}
valid_counts = {}

# Calculate elasticity for the 'total' column
valid_total_elasticity = price_elasticity_category['elasticity_total'].dropna()
elasticity['total'] = valid_total_elasticity.mean()
standard_deviations['total'] = valid_total_elasticity.std()
t_statistics['total'] = '-'
p_values['total'] = '-'
valid_counts['total'] = len(valid_total_elasticity)

# Calculate elasticity for each column except 'total'
columns = validator_exit_percentage_category.columns.drop('slot')
for col in columns:
    price_elasticity_category[f'elasticity_{col}'] = price_elasticity_category[col] / price_elasticity_category['price_pct_change']
    
    # Replace infinite values with NaN
    price_elasticity_category[f'elasticity_{col}'] = price_elasticity_category[f'elasticity_{col}'].replace([np.inf, -np.inf], np.nan)
    
    # Drop NaN values for calculation purposes
    valid_elasticity = price_elasticity_category[f'elasticity_{col}'].dropna()
    
    elasticity[col] = valid_elasticity.mean()
    standard_deviations[col] = valid_elasticity.std()
    
    # Perform a two-sample t-test against the 'total' elasticity
    t_stat, p_value = stats.ttest_ind(valid_elasticity, valid_total_elasticity, equal_var=False)
    t_statistics[col] = t_stat
    p_values[col] = p_value
    
    # Store the number of valid rows
    valid_counts[col] = len(valid_elasticity)

# Print the results in the requested format
print("Elasticity Analysis Results:")
print(f'total: Mean Elasticity = {elasticity["total"]}, t(Mean) = {t_statistics["total"]}, SD = {standard_deviations["total"]}, N = {valid_counts["total"]}, p-value = {p_values["total"]}')
for col in columns:
    print(f'{col}: Mean Elasticity = {elasticity[col]}, t(Mean) = {t_statistics[col]}, SD = {standard_deviations[col]}, N = {valid_counts[col]}, p-value = {p_values[col]}')

# Display the DataFrame with elasticity columns
print(price_elasticity_category)

Elasticity Analysis Results:
total: Mean Elasticity = -0.2077412695628768, t(Mean) = 0.0, SD = 5.70050321762948, N = 387, p-value = 1.0
CEX: Mean Elasticity = -1.0141667407825998, t(Mean) = -0.7674604799131861, SD = 19.869545926918384, N = 387, p-value = 0.4432110600265867
Liquid Restaking: Mean Elasticity = -0.020107233900713565, t(Mean) = 0.6459677985618866, SD = 0.39555585059307097, N = 387, p-value = 0.5186803868643363
Liquid Staking: Mean Elasticity = 0.028997895681425266, t(Mean) = 0.6937548388859676, SD = 3.545312180671527, N = 387, p-value = 0.48808544388327346
Solo Stakers: Mean Elasticity = -0.3923096910217833, t(Mean) = -0.39550901628062796, SD = 7.195972696413952, N = 387, p-value = 0.6925822571245963
Staking Pools: Mean Elasticity = 0.6519993528271224, t(Mean) = 0.6750280975341603, SD = 24.398290472148336, N = 387, p-value = 0.5000223885614099
Unidentified: Mean Elasticity = 0.10620835254154555, t(Mean) = 0.8894739865528891, SD = 3.964503658342797, N = 387, p-value = 0.374

In [34]:
# Merge the two DataFrames on the slot column
price_elasticity_pool = pd.merge(validator_exit_percentage_pool, aave[['slot', 'price_pct_change']], on='slot')

# Drop rows with infinite values
price_elasticity_pool.replace([np.inf, -np.inf], np.nan, inplace=True)

# Calculate elasticity for 'total' first
price_elasticity_pool['elasticity_total'] = price_elasticity_pool['total'] / price_elasticity_pool['price_pct_change']
price_elasticity_pool['elasticity_total'] = price_elasticity_pool['elasticity_total'].replace([np.inf, -np.inf], np.nan)

# Initialize dictionaries to store elasticity values, standard deviations, t-statistics, p-values, and number of valid rows (N)
elasticity = {}
standard_deviations = {}
t_statistics = {}
p_values = {}
valid_counts = {}

# Calculate elasticity for the 'total' column
valid_total_elasticity = price_elasticity_pool['elasticity_total'].dropna()
elasticity['total'] = valid_total_elasticity.mean()
standard_deviations['total'] = valid_total_elasticity.std()
t_statistics['total'] = '-'
p_values['total'] = '-'
valid_counts['total'] = len(valid_total_elasticity)

# Calculate elasticity for each column except 'total'
columns = ['Lido', 'Coinbase', 'Binance', 'Rocketpool', 'Kraken', 'OKX', 'Bitcoin Suisse', 'Ledger Live', 'Ether.Fi', 'Mantle', 'Other Stakers']
for col in columns:
    price_elasticity_pool[f'elasticity_{col}'] = price_elasticity_pool[col] / price_elasticity_pool['price_pct_change']
    
    # Replace infinite values with NaN
    price_elasticity_pool[f'elasticity_{col}'] = price_elasticity_pool[f'elasticity_{col}'].replace([np.inf, -np.inf], np.nan)
    
    # Drop NaN values for calculation purposes
    valid_elasticity = price_elasticity_pool[f'elasticity_{col}'].dropna()
    
    elasticity[col] = valid_elasticity.mean()
    standard_deviations[col] = valid_elasticity.std()
    
    # Perform a two-sample t-test against the 'total' elasticity
    t_stat, p_value = stats.ttest_ind(valid_elasticity, valid_total_elasticity, equal_var=False)
    t_statistics[col] = t_stat
    p_values[col] = p_value
    
    # Store the number of valid rows
    valid_counts[col] = len(valid_elasticity)

# Print the results in the requested format
print("Elasticity Analysis Results:")
print(f'total: Mean Elasticity = {elasticity["total"]}, t(Mean) = {t_statistics["total"]}, SD = {standard_deviations["total"]}, N = {valid_counts["total"]}, p-value = {p_values["total"]}')
for col in columns:
    print(f'{col}: Mean Elasticity = {elasticity[col]}, t(Mean) = {t_statistics[col]}, SD = {standard_deviations[col]}, N = {valid_counts[col]}, p-value = {p_values[col]}')

# Display the DataFrame with elasticity columns
print(price_elasticity_pool)

Elasticity Analysis Results:
total: Mean Elasticity = -0.2077412695628768, t(Mean) = -, SD = 5.70050321762948, N = 387, p-value = -
Lido: Mean Elasticity = 0.17274758514199168, t(Mean) = 1.144243458827078, SD = 3.208708175848818, N = 387, p-value = 0.252972745634548
Coinbase: Mean Elasticity = -0.048114623142164394, t(Mean) = 0.5384013017855759, SD = 1.233824537634038, N = 387, p-value = 0.5905838337876403
Binance: Mean Elasticity = -6.788464417218962, t(Mean) = -0.991594451748878, SD = 130.43094138500513, N = 387, p-value = 0.32201421000441166
Rocketpool: Mean Elasticity = -0.09593286988103768, t(Mean) = 0.3269667772663551, SD = 3.571806935746409, N = 387, p-value = 0.7437985019105311
Kraken: Mean Elasticity = 0.2371339527596201, t(Mean) = 1.1317481659345794, SD = 5.225171817147683, N = 387, p-value = 0.25809439479113117
OKX: Mean Elasticity = -2.9989761481247297, t(Mean) = -0.9883602054296668, SD = 55.26349287697797, N = 387, p-value = 0.32358250876913264
Bitcoin Suisse: Mean Elastic